# External Data Testing Notebook

Notebook ini digunakan untuk me-load model yang sudah dilatih dan melakukan prediksi massal (batch inference) pada dataset CSV baru yang belum pernah dilihat oleh model (berbeda dengan data training).

In [ ]:
import sys
import os

# Tambahkan parent directory ke system path agar bisa meng-import module dari folder src
sys.path.append(os.path.abspath('../'))

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from gensim.models import Word2Vec
from src.utils.config_parser import load_config
from src.encoding.kmer import build_kmers, create_seqs_embedding

In [ ]:
# 1. Load Konfigurasi
config = load_config('../configs/config.yaml')
ksize = config['encoding']['kmer_size']
vector_size = config['encoding']['word2vec']['vector_size']

# 2. Load Model Word2Vec
w2v_path = os.path.join('../', config['paths']['weights_dir'], 'word2vec.model')
w2v_model = Word2Vec.load(w2v_path)

# 3. Load Model Keras (LSTM atau Attention LSTM)
model_path = os.path.join('../', config['paths']['weights_dir'], 'lstm_model.h5')
model = keras.models.load_model(model_path)

print("Models loaded successfully!")

In [ ]:
# 4. Load Dataset Baru (CSV)
testing_data_path = "../data/data_test.csv"

if testing_data_path.endswith('.csv'):
    df_test = pd.read_csv(testing_data_path)
else:
    df_test = pd.read_excel(testing_data_path)

print(f"Total data testing: {len(df_test)}")
df_test.head()

In [ ]:
# 5. Ekstrak K-mers untuk seluruh data
kmers_list = []
for seq in df_test['sequence']:
    kmer = build_kmers(seq, ksize)
    kmers_list.append(kmer)

# 6. Buat Embeddings (Word2Vec) dan Normalisasi
embeddings = create_seqs_embedding(w2v_model, kmers_list, vector_size)

# 7. Reshape untuk model LSTM (samples, timesteps, features)
embeddings_reshaped = np.expand_dims(embeddings, axis=-1)

# 8. Prediksi
probabilitas = model.predict(embeddings_reshaped)

# 9. Konversi probabilitas ke class 0 atau 1
prediksi_kelas = np.where(probabilitas > 0.5, 1, 0)

df_test['probabilitas'] = probabilitas
df_test['prediksi_kelas'] = prediksi_kelas

df_test.head()

In [ ]:
# 10. Simpan hasil prediksi ke file CSV baru
output_path = "../outputs/hasil_prediksi_testing.csv"
df_test.to_csv(output_path, index=False)

print(f"Prediksi selesai! Hasil disimpan di {output_path}")